# Persian Webpage Topic Classification

A multi-class classification project for Persian webpages using **TF-IDF** and **LinearSVC**.

The official evaluation metric is **weighted F1-score**. This notebook compares a richer feature set
(text + URL + domain) with a simpler **text-only** pipeline and selects the better validation model.

**Best observed validation result:** `Weighted F1 = 0.8167` with the text-only model.

## 1. Setup

The dataset files are not included in this repository. Place these files next to the notebook before running it:

- `yektanet_train.csv`
- `yektanet_test.csv`

The experiments use `random_state=42` for reproducibility.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.svm import LinearSVC

## 2. Load the data

In [ ]:
train = pd.read_csv("yektanet_train.csv")
test = pd.read_csv("yektanet_test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (4789, 9)
Test shape: (417, 8)


In [ ]:
train.head()

              category                                        description  \
0        کتاب و ادبیات  از شوبنده ها: جستجو معنی "از شوبنده ها" در فره...   
1       تجارت و اقتصاد  بیت‌کوین کش یک ارز مجازی مشهور است و بیت‌کوین ...   
2                سلامت  نوبت دهی دکتر مهناز عابدینی متخصص رادیولوژی و ...   
3  تکنولوژی و کامپبوتر  نرم افزار Geph برای اندروید یک پلت‌فرم چندسکوی...   
4  تکنولوژی و کامپبوتر  سری جدید تلویزیون‌های هوشمند سامسونگ که با نام...   

                                        text_content  \
0      معنی از شوبنده ها | جدول یاب از شوبنده ها 381   
1  عکس بیت‌کوین کش برای پروفایل عکس و والپیپرهای ...   
2  دکتر مهناز عابدینی متخصص رادیولوژی و سونوگرافی...   
3  دانلود تحریم‌گذر Geph برای اندروید خانه/اندروی...   
4  ترفندهای پرکاربرد تلویزیون‌‌های هوشمند سامسونگ...   

                                               title  \
0                       معنی از شوبنده ها | جدول یاب   
1                       عکس بیت‌کوین کش برای پروفایل   
2  دکتر مهناز عابدینی متخصص رادی

## 3. Dataset inspection

In [ ]:
train['category'].value_counts()

category
سلامت                  614
ورزش                   514
حقوق و دولت و سیاست    486
هنر و سرگرمی           410
موسیقی                 314
تکنولوژی و کامپبوتر    287
تجارت و اقتصاد         283
فیلم و سینما           239
خودرو                  237
اجتماعی                209
سفر و گردشگری          182
غذا و نوشیدنی          171
مذهبی                  160
مسکن                   131
خانه و باغبانی         128
مد و زیبایی            118
کتاب و ادبیات           83
تحصیلات                 79
اشتغال                  47
علم و دانش              34
خانواده                 34
حیوانات خانگی           29
Name: count, dtype: int64

In [ ]:
train.duplicated().sum()

0

The dataset contains **22 classes** with noticeable class imbalance.  
Weighted F1 is the primary metric, while macro F1 is also reported to show how the model behaves on smaller classes.

## 4. Missing values and train/validation split

In [ ]:
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

missing_summary = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_percent": train.isna().mean().mul(100),
})

missing_summary

              missing_count  missing_percent
category                   0         0.000000
description               46         0.960535
text_content               0         0.000000
title                      0         0.000000
h1                       358         7.475465
h2                      1439        30.048027
url                        0         0.000000
domain                     5         0.104406

In [ ]:
X = train.drop(columns=["category"])
y = train["category"]

x_train, x_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Train split:", x_train.shape)
print("Validation split:", x_val.shape)

Train split: (3831, 7)
Validation split: (958, 7)


## 5. Lightweight Persian text preprocessing

A heavy Persian tokenizer was intentionally avoided because it made experimentation much slower.
Instead, the pipeline uses a lightweight normalizer and word n-grams.

The normalizer:
- standardizes common Arabic/Persian character variants,
- removes diacritics,
- converts zero-width non-joiners to spaces,
- maps numeric sequences to a shared `NUM` token,
- normalizes whitespace.

In [ ]:
main_text_cols = [
    "description",
    "text_content",
    "title",
    "h1",
    "h2",
]

url_col = ["url"]
domain_col = ["domain"]


def combine_text_columns(X):
    return np.array([
        " ".join(map(str, row))
        for row in X
    ])


def flatten_column(X):
    return np.asarray(X).ravel()

In [ ]:
_translation_table = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "ة": "ه",
})

_diacritics_pattern = re.compile(
    r"[\u064B-\u065F\u0670\u06D6-\u06ED]"
)
_number_pattern = re.compile(r"[0-9۰-۹٠-٩]+")
_space_pattern = re.compile(r"\s+")


def normalize_fa_text(text):
    text = str(text).lower()
    text = text.translate(_translation_table)
    text = _diacritics_pattern.sub("", text)
    text = text.replace("\u200c", " ")
    text = _number_pattern.sub(" NUM ", text)
    text = _space_pattern.sub(" ", text)
    return text.strip()

In [ ]:
main_text_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="",
        ),
    ),
    (
        "combine",
        FunctionTransformer(
            combine_text_columns,
            validate=False,
        ),
    ),
    (
        "tfidf",
        TfidfVectorizer(
            preprocessor=normalize_fa_text,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=30000,
            sublinear_tf=True,
            dtype=np.float32,
        ),
    ),
])

## 6. Experiment A — Text + URL + domain

In [ ]:
url_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="",
        ),
    ),
    (
        "flatten",
        FunctionTransformer(
            flatten_column,
            validate=False,
        ),
    ),
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_features=10000,
            sublinear_tf=True,
            dtype=np.float32,
        ),
    ),
])

domain_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

full_preprocessor = ColumnTransformer([
    ("main_text", main_text_pipeline, main_text_cols),
    ("url", url_pipeline, url_col),
    ("domain", domain_pipeline, domain_col),
])

### Fast classifier tuning

To keep runtime practical, the TF-IDF representation is computed once and the search tunes only the `LinearSVC`
classifier. The selected classifier is then checked on the untouched validation split.

In [ ]:
X_train_full = full_preprocessor.fit_transform(x_train)
X_val_full = full_preprocessor.transform(x_val)

print("Feature matrix:", X_train_full.shape)

Feature matrix: (3831, 40750)


In [ ]:
param_grid = {
    "C": [0.25, 0.5, 1, 2, 4],
    "class_weight": [None, "balanced"],
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

grid_search = GridSearchCV(
    estimator=LinearSVC(
        random_state=42,
        max_iter=5000,
    ),
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

_ = grid_search.fit(X_train_full, y_train)

In [ ]:
print("Best params:", grid_search.best_params_)
print("Best CV Weighted F1:", grid_search.best_score_)

full_tuned_pred = grid_search.best_estimator_.predict(X_val_full)

full_tuned_weighted_f1 = f1_score(
    y_val,
    full_tuned_pred,
    average="weighted",
)

print("Validation Weighted F1:", full_tuned_weighted_f1)

Best params: {'C': 1, 'class_weight': 'balanced'}
Best CV Weighted F1: 0.7780776743637036
Validation Weighted F1: 0.7970773328372261


## 7. Experiment B — Text only

In [ ]:
text_only_preprocessor = ColumnTransformer([
    ("main_text", main_text_pipeline, main_text_cols),
])

text_only_model = Pipeline([
    ("preprocessor", text_only_preprocessor),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            random_state=42,
            max_iter=5000,
        ),
    ),
])

text_only_model.fit(x_train, y_train)

text_only_pred = text_only_model.predict(x_val)

text_only_weighted_f1 = f1_score(
    y_val,
    text_only_pred,
    average="weighted",
)

text_only_macro_f1 = f1_score(
    y_val,
    text_only_pred,
    average="macro",
)

print("Text-only Weighted F1:", text_only_weighted_f1)
print("Text-only Macro F1:", text_only_macro_f1)

Text-only Weighted F1: 0.8167341155343786
Text-only Macro F1: 0.7215495908544095


### Model comparison

| Model | Validation Weighted F1 | Validation Macro F1 |
|---|---:|---:|
| Text + URL + domain (tuned classifier) | 0.7971 | — |
| **Text only** | **0.8167** | **0.7215** |

The text-only pipeline performs better while also being simpler and cheaper to run.
Its weighted F1 is about **0.0197 higher** than the tuned richer-feature model, so it is selected as the final model.

## 8. Error analysis

In [ ]:
print(
    classification_report(
        y_val,
        text_only_pred,
        zero_division=0,
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    text_only_pred,
    xticks_rotation=90,
    ax=ax,
)

plt.title("Text-only Model — Confusion Matrix")
plt.tight_layout()
plt.show()

## 9. Final training and test inference

The selected text-only pipeline is refit on all labeled training rows before generating test predictions.
The generated `submission.csv` is ignored by Git because it is an output artifact rather than source code.

In [ ]:
X_full = pd.concat([x_train, x_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

final_model = Pipeline([
    ("preprocessor", text_only_preprocessor),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            random_state=42,
            max_iter=5000,
        ),
    ),
])

final_model.fit(X_full, y_full);

In [ ]:
test_pred = final_model.predict(test)

submission = pd.DataFrame({
    "category": test_pred,
})

print("Submission shape:", submission.shape)
submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

## 10. Takeaways

- Lightweight normalization + TF-IDF + LinearSVC is a strong baseline for this Persian text classification task.
- Adding URL and domain features did **not** improve validation performance.
- The simpler text-only model achieved the best observed validation score: **Weighted F1 = 0.8167**.
- Avoiding a heavy tokenizer kept experimentation much faster.